# Poisson regression baseline

Fit the preprocessing pipeline on January–August 2024 and inspect September predictions.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))


In [ ]:
X = df.copy()

X_train = X[X["tpep_pickup_datetime"].dt.month <= 8]
X_val = X[(X["tpep_pickup_datetime"].dt.month > 8)&(X["tpep_pickup_datetime"].dt.month <= 9)]
X_test = X[X["tpep_pickup_datetime"].dt.month > 9]
y_train = X_train.trip_count
y_val = X_val.trip_count
y_test = X_test.trip_count


In [ ]:
train = X_train.drop(columns = ["trip_count"])
val = X_val.drop(columns = ["trip_count"])
test = X_test.drop(columns = ["trip_count"])

train = train.drop(columns = ["tpep_pickup_datetime"])
val = val.drop(columns = ["tpep_pickup_datetime"])
test = test.drop(columns = ["tpep_pickup_datetime"])


In [ ]:
train_no_miss = train.dropna( subset = ["lag_1h", "lag_24h", "lag_1w", "roll_mean_3", "roll_mean_6", "roll_mean_24"]) #drop rows with missing values (2.9% of training data) to fit the Poisson regressor.
y_train_no_miss = y_train.loc[train_no_miss.index]


In [ ]:
categorical = ["PULocationID", "hours"]
numerical = [
    col for col in train_no_miss.columns
    if col not in categorical
]

preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
Poisson = make_pipeline(
    preprocessor,
    linear_model.PoissonRegressor(max_iter = 500, alpha = 0.05)
)

Poisson.fit(train_no_miss, y_train_no_miss)


In [ ]:
y_pred = Poisson.predict(val)

mae = mean_absolute_error(y_val, y_pred)
rsme = root_mean_squared_error(y_val, y_pred)
mae, rsme


## Inspect prediction errors and demand distributions


In [ ]:
m_pred = y_pred.mean()
med_pred = np.median(y_pred)
max_pred = np.max(y_pred)
p95_pred = np.percentile(y_pred, 95)
p99_pred = np.percentile(y_pred, 99)

print("Validation performance")
print(f"MAE:  {mean_absolute_error(y_val, y_pred):.4f}")
print(f"RMSE: {root_mean_squared_error(y_val, y_pred):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {m_pred:.4f}")
print(f"Median:          {med_pred:.4f}")
print(f"95th percentile: {p95_pred:.4f}")
print(f"99th percentile: {p99_pred:.4f}")
print(f"Maximum:         {max_pred:.4f}")


In [ ]:
m_val = y_val.mean()
med_val = np.median(y_val)
max_val = np.max(y_val)
p95_val = np.percentile(y_val, 95)
p99_val = np.percentile(y_val, 99)

print("Actual validation pickups per zone per hour")
print(f"Mean:            {m_val:.4f}")
print(f"Median:          {med_val:.4f}")
print(f"95th percentile: {p95_val:.4f}")
print(f"99th percentile: {p99_val:.4f}")
print(f"Maximum:         {max_val:.4f}")


In [ ]:
# Match predictions to validation rows before aggregating.
pickup_comparison = X_val[["tpep_pickup_datetime", "PULocationID"]].copy()
pickup_comparison["Actual"] = y_val.loc[pickup_comparison.index]
pickup_comparison["Predicted"] = np.asarray(y_pred)

hourly_comparison = pickup_comparison.groupby("tpep_pickup_datetime")[["Actual", "Predicted"]].sum().sort_index()
zone_comparison = pickup_comparison.groupby("PULocationID")[["Actual", "Predicted"]].sum().sort_index()

fig, ax = plt.subplots(figsize = (14, 4))
ax.plot(hourly_comparison.index, hourly_comparison["Actual"], label = "Actual", color = "tab:blue")
ax.plot(hourly_comparison.index, hourly_comparison["Predicted"], label = "Predicted", color = "tab:orange")
ax.set_title("Total pickups across all zones at each validation hour")
ax.set_xlabel("Pickup date and hour")
ax.set_ylabel("Total pickups")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize = (14, 4))
zone_positions = np.arange(len(zone_comparison))
ax.bar(zone_positions - 0.2, zone_comparison["Actual"], width = 0.4, label = "Actual", color = "tab:blue")
ax.bar(zone_positions + 0.2, zone_comparison["Predicted"], width = 0.4, label = "Predicted", color = "tab:orange")
ax.set_xticks(zone_positions[::10], zone_comparison.index[::10], rotation = 45)
ax.set_title("Total pickups per zone over the validation period")
ax.set_xlabel("Pickup zone ID")
ax.set_ylabel("Total pickups")
ax.legend()
fig.tight_layout()
plt.show()


### Inspect the large overpredictions in zone 131


In [ ]:
high_pred = sorted(y_pred.astype(int).tolist(), reverse = True)[:10]
high_val = sorted(y_val, reverse = True)[:10]
high_pred, high_val


In [ ]:
import matplotlib.pyplot as plt

# Compare the first seven days of the validation period for zone 131.
week_start = X_val["tpep_pickup_datetime"].min().normalize()
week_end = week_start + pd.Timedelta(days = 7)
zone_week = X_val[["tpep_pickup_datetime", "PULocationID"]].copy()
zone_week["Actual"] = y_val.loc[zone_week.index]
zone_week["Predicted"] = np.asarray(y_pred)
zone_week = zone_week.loc[
    (zone_week["PULocationID"] == 131)
    & (zone_week["tpep_pickup_datetime"] >= week_start)
    & (zone_week["tpep_pickup_datetime"] < week_end)
].sort_values("tpep_pickup_datetime")

fig, ax = plt.subplots(figsize = (14, 4))
ax.plot(zone_week["tpep_pickup_datetime"], zone_week["Actual"], color = "tab:blue", label = "Actual")
ax.plot(zone_week["tpep_pickup_datetime"], zone_week["Predicted"], color = "tab:orange", label = "Predicted")
ax.set_title(f"Zone 131: hourly demand, {week_start:%d %b %Y}–{week_end - pd.Timedelta(days = 1):%d %b %Y}")
ax.set_xlabel("Pickup date and hour")
ax.set_ylabel("Pickups per hour")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


The tested Poisson regression specification performs poorly in this experiment, including large overpredictions in some zones.
